# 03 - LLM evaluation

Compare two prompt variants on the same retrieval backend and grade the
generated answers with `judge.evaluate_relevance` (LLM-as-judge).

In [ ]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from assistant import build_searcher
from llmclient import get_llm_client
from metrics import RAGWithMetrics
from judge import evaluate_relevance

client = get_llm_client()
searcher = build_searcher('hybrid_rerank_rewrite', llm_client=client)

with open('ground_truth.json') as f:
    gt = json.load(f)

In [ ]:
PROMPT_A = '''You are a wildlife tracking research assistant. Answer using ONLY the provided study context. Cite study name and id. If not present, reply "I don't know based on the tracked studies I have access to."'''

PROMPT_B = '''You are a helpful research librarian for animal tracking studies. Give a concise, information-dense answer that ONLY uses the provided study context. Include the study id in parentheses after each study name. When information is missing, say "I don't know based on the tracked studies I have access to."'''

def build(instr):
    return RAGWithMetrics(search_engine=searcher, llm_client=client, instructions=instr)

rag_a = build(PROMPT_A)
rag_b = build(PROMPT_B)

from collections import Counter

SAMPLE = gt[:20]

def run(rag):
    verdicts = []
    for row in SAMPLE:
        q = row['question']
        a = rag.rag(q)
        rel, expl = evaluate_relevance(q, a)
        verdicts.append((rel, q, a, expl))
    return verdicts

va = run(rag_a)
vb = run(rag_b)

print('Prompt A:', Counter([r for r,*_ in va]))
print('Prompt B:', Counter([r for r,*_ in vb]))

In [ ]:
import pandas as pd
rows = []
for name, v in [('A', va), ('B', vb)]:
    c = Counter([r for r,*_ in v])
    total = sum(c.values()) or 1
    rows.append({
        'prompt': name,
        'RELEVANT': c.get('RELEVANT', 0) / total,
        'PARTLY_RELEVANT': c.get('PARTLY_RELEVANT', 0) / total,
        'NON_RELEVANT': c.get('NON_RELEVANT', 0) / total,
    })
df = pd.DataFrame(rows).set_index('prompt')
df.to_csv('llm_eval.csv')
df